In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("INFO602_Final_Project").getOrCreate()
spark

In [6]:
import requests

od_url = "https://data.virginia.gov/dataset/40616839-ab5c-4bdd-8322-3217d035e287/resource/1513876b-3534-4644-9b5e-3f279dd1b8a6/download/vdh-pud-overdose-ed-visits-by-year-and-geography.csv"

resp = requests.get(od_url)
resp.raise_for_status()

with open("vdh-pud-overdose-ed-visits-by-year-and-geography.csv", "wb") as f:
    f.write(resp.content)

print("Downloaded, size:", len(resp.content))

Downloaded, size: 287153


In [7]:
od_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("vdh-pud-overdose-ed-visits-by-year-and-geography.csv")

od_df.printSchema()
od_df.show(5, truncate=False)

root
 |-- Data Extract Date: string (nullable = true)
 |-- Overdose ED Visit Year: integer (nullable = true)
 |-- Overdose ED Visit Drug Type: string (nullable = true)
 |-- Overdose ED Visit Patient Geography Level: string (nullable = true)
 |-- Overdose ED Visit Patient Geography Name: string (nullable = true)
 |-- Overdose ED Visit Patient FIPS: string (nullable = true)
 |-- Combined Locality: string (nullable = true)
 |-- Overdose ED Visit Count: string (nullable = true)
 |-- Overdose ED Visit Rate per 10,000 visits: double (nullable = true)

+-----------------+----------------------+---------------------------+-----------------------------------------+---------------------------------------------+------------------------------+-----------------+-----------------------+----------------------------------------+
|Data Extract Date|Overdose ED Visit Year|Overdose ED Visit Drug Type|Overdose ED Visit Patient Geography Level|Overdose ED Visit Patient Geography Name     |Overdose ED Visit

In [10]:
from pyspark.sql.functions import col

od_2024 = od_df.filter(col("Overdose ED Visit Year") == 2024)

In [11]:
od_local = od_2024.filter(col("Overdose ED Visit Patient Geography Level") == "Locality")

In [13]:
od_local_sel = od_local.select(
    col("Overdose ED Visit Patient Geography Name").alias("Locality"),
    col("Overdose ED Visit Year").alias("year"),
    col("Overdose ED Visit Rate per 10,000 visits").alias("od_rate")
)

In [14]:
od_local_sel.show(10, truncate=False)

+-------------------------------------------------------+----+-------+
|Locality                                               |year|od_rate|
+-------------------------------------------------------+----+-------+
|Middlesex                                              |2024|1.4    |
|Dickenson                                              |2024|26.7   |
|Alleghany County and Covington City                    |2024|20.8   |
|Westmoreland                                           |2024|0.0    |
|Fluvanna                                               |2024|1.0    |
|Rockbridge County, Buena Vista City, and Lexington City|2024|1.1    |
|Botetourt                                              |2024|44.1   |
|Craig                                                  |2024|30.1   |
|Fairfax County, Fairfax City, and Falls Church City    |2024|19.6   |
|Louisa                                                 |2024|4.2    |
+-------------------------------------------------------+----+-------+
only s

In [15]:
od_2024 = od_local_sel

In [16]:
acs_df = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv("/content/ACSDT1Y2024.B17018-Data.csv")

In [18]:
acs_df.printSchema()

root
 |-- GEO_ID: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- B17018_001E: string (nullable = true)
 |-- B17018_001M: string (nullable = true)
 |-- B17018_002E: string (nullable = true)
 |-- B17018_002M: string (nullable = true)
 |-- B17018_003E: string (nullable = true)
 |-- B17018_003M: string (nullable = true)
 |-- B17018_004E: string (nullable = true)
 |-- B17018_004M: string (nullable = true)
 |-- B17018_005E: string (nullable = true)
 |-- B17018_005M: string (nullable = true)
 |-- B17018_006E: string (nullable = true)
 |-- B17018_006M: string (nullable = true)
 |-- B17018_007E: string (nullable = true)
 |-- B17018_007M: string (nullable = true)
 |-- B17018_008E: string (nullable = true)
 |-- B17018_008M: string (nullable = true)
 |-- B17018_009E: string (nullable = true)
 |-- B17018_009M: string (nullable = true)
 |-- B17018_010E: string (nullable = true)
 |-- B17018_010M: string (nullable = true)
 |-- B17018_011E: string (nullable = true)
 |-- B17018_011M: 

In [19]:
acs_df.select("NAME", "B17018_001E", "B17018_002E").show(5, truncate=False)

+-----------------------+----------------+-------------------------------------------------------------------+
|NAME                   |B17018_001E     |B17018_002E                                                        |
+-----------------------+----------------+-------------------------------------------------------------------+
|Geographic Area Name   |Estimate!!Total:|Estimate!!Total:!!Income in the past 12 months below poverty level:|
|Baldwin County, Alabama|72891           |5260                                                               |
|Calhoun County, Alabama|29688           |4864                                                               |
|Cullman County, Alabama|26010           |2949                                                               |
|DeKalb County, Alabama |17780           |3348                                                               |
+-----------------------+----------------+-------------------------------------------------------------------+
o

In [23]:
from pyspark.sql.functions import col, when, expr

# Drop the header row where NAME == "Geographic Area Name"
acs_no_header = acs_df.filter(col("NAME") != "Geographic Area Name")

# Cast to numeric using try_cast
acs_cast = acs_no_header.select(
    col("GEO_ID"),
    col("NAME"),
    expr("try_cast(B17018_001E AS DOUBLE)").alias("total_families"),
    expr("try_cast(B17018_002E AS DOUBLE)").alias("poverty_families")
)

# Compute poverty_percent safely (avoid division by zero and handle nulls)
acs_poverty = acs_cast.withColumn(
    "poverty_percent",
    when(col("total_families").isNotNull() & (col("total_families") > 0),
         col("poverty_families") / col("total_families") * 100).otherwise(None)
)

acs_poverty.select("NAME", "total_families", "poverty_families", "poverty_percent") \
    .show(10, truncate=False)

+--------------------------+--------------+----------------+------------------+
|NAME                      |total_families|poverty_families|poverty_percent   |
+--------------------------+--------------+----------------+------------------+
|Baldwin County, Alabama   |72891.0       |5260.0          |7.216254407265643 |
|Calhoun County, Alabama   |29688.0       |4864.0          |16.38372406359472 |
|Cullman County, Alabama   |26010.0       |2949.0          |11.337946943483274|
|DeKalb County, Alabama    |17780.0       |3348.0          |18.830146231721034|
|Elmore County, Alabama    |NULL          |NULL            |NULL              |
|Etowah County, Alabama    |28116.0       |3711.0          |13.19889031156637 |
|Houston County, Alabama   |28833.0       |3876.0          |13.442929976069088|
|Jefferson County, Alabama |166513.0      |16290.0         |9.783019944388727 |
|Lauderdale County, Alabama|26175.0       |1955.0          |7.468958930276982 |
|Lee County, Alabama       |43519.0     

In [28]:
from pyspark.sql.functions import upper, trim, expr

acs_va = acs_poverty.filter("NAME LIKE '%, Virginia'")

acs_va_clean = acs_va.withColumn(
    "county_clean",
    upper(trim(expr("substr(NAME, 1, length(NAME) - length(', Virginia'))")))
)
acs_va_clean.select("NAME", "county_clean", "poverty_percent") \
    .show(10, truncate=False)

+-----------------------------+-------------------+------------------+
|NAME                         |county_clean       |poverty_percent   |
+-----------------------------+-------------------+------------------+
|Albemarle County, Virginia   |ALBEMARLE COUNTY   |NULL              |
|Arlington County, Virginia   |ARLINGTON COUNTY   |NULL              |
|Augusta County, Virginia     |AUGUSTA COUNTY     |NULL              |
|Bedford County, Virginia     |BEDFORD COUNTY     |NULL              |
|Chesterfield County, Virginia|CHESTERFIELD COUNTY|4.673505176022476 |
|Fairfax County, Virginia     |FAIRFAX COUNTY     |3.9425849102418966|
|Fauquier County, Virginia    |FAUQUIER COUNTY    |NULL              |
|Frederick County, Virginia   |FREDERICK COUNTY   |NULL              |
|Hanover County, Virginia     |HANOVER COUNTY     |5.3908355795148255|
|Henrico County, Virginia     |HENRICO COUNTY     |7.605108499095841 |
+-----------------------------+-------------------+------------------+
only s

In [29]:
from pyspark.sql.functions import upper, trim

od_2024_clean = od_2024.withColumn(
    "county_clean",
    upper(trim("Locality"))
)

od_2024_clean.select("Locality", "county_clean", "od_rate") \
    .show(10, truncate=False)

+-------------------------------------------------------+-------------------------------------------------------+-------+
|Locality                                               |county_clean                                           |od_rate|
+-------------------------------------------------------+-------------------------------------------------------+-------+
|Middlesex                                              |MIDDLESEX                                              |1.4    |
|Dickenson                                              |DICKENSON                                              |26.7   |
|Alleghany County and Covington City                    |ALLEGHANY COUNTY AND COVINGTON CITY                    |20.8   |
|Westmoreland                                           |WESTMORELAND                                           |0.0    |
|Fluvanna                                               |FLUVANNA                                               |1.0    |
|Rockbridge County, Buen

In [30]:
from pyspark.sql.functions import upper, trim, instr

# Start from od_2024 (Locality, year, od_rate)
od_simple = od_2024_clean.filter(
    (instr("Locality", "City") == 0) & (instr("Locality", "and") == 0)
)

# Now map to "<COUNTY> COUNTY" style
from pyspark.sql.functions import concat, lit

od_simple = od_simple.withColumn(
    "county_join_key",
    concat(upper(trim("Locality")), lit(" COUNTY"))
)

od_simple.select("Locality", "county_join_key", "od_rate") \
    .show(20, truncate=False)

+---------+----------------+-------+
|Locality |county_join_key |od_rate|
+---------+----------------+-------+
|Middlesex|MIDDLESEX COUNTY|1.4    |
|Dickenson|DICKENSON COUNTY|26.7   |
|Fluvanna |FLUVANNA COUNTY |1.0    |
|Botetourt|BOTETOURT COUNTY|44.1   |
|Craig    |CRAIG COUNTY    |30.1   |
|Louisa   |LOUISA COUNTY   |4.2    |
|Lynchburg|LYNCHBURG COUNTY|32.7   |
|Surry    |SURRY COUNTY    |2.8    |
|Sussex   |SUSSEX COUNTY   |2.2    |
|Hampton  |HAMPTON COUNTY  |38.7   |
|Poquoson |POQUOSON COUNTY |0.0    |
|Buchanan |BUCHANAN COUNTY |83.1   |
|Patrick  |PATRICK COUNTY  |4.7    |
|Nottoway |NOTTOWAY COUNTY |20.8   |
|York     |YORK COUNTY     |1.2    |
|Bath     |BATH COUNTY     |8.9    |
|Culpeper |CULPEPER COUNTY |0.0    |
|Fauquier |FAUQUIER COUNTY |42.6   |
|Floyd    |FLOYD COUNTY    |3.2    |
|Suffolk  |SUFFOLK COUNTY  |1.3    |
+---------+----------------+-------+
only showing top 20 rows


In [32]:
acs_join = acs_va_clean.select(
    "county_clean",
    "poverty_percent"
)
acs_join_nonnull = acs_join.filter("poverty_percent IS NOT NULL")
joined = od_simple.join(
    acs_join_nonnull,
    od_simple["county_join_key"] == acs_join_nonnull["county_clean"],
    how="inner"
)

joined.select("Locality", "od_rate", "poverty_percent") \
    .show(20, truncate=False)

+------------+-------+------------------+
|Locality    |od_rate|poverty_percent   |
+------------+-------+------------------+
|Hanover     |40.5   |5.3908355795148255|
|Spotsylvania|1.1    |4.9209481351555455|
|Spotsylvania|0.1    |4.9209481351555455|
|Hanover     |1.1    |5.3908355795148255|
|Henrico     |58.7   |7.605108499095841 |
|Loudoun     |38.1   |3.2100979202925304|
|Spotsylvania|30.2   |4.9209481351555455|
|Henrico     |1.7    |7.605108499095841 |
|Loudoun     |0.0    |3.2100979202925304|
|Hanover     |17.4   |5.3908355795148255|
|Henrico     |29.6   |7.605108499095841 |
|Loudoun     |12.9   |3.2100979202925304|
|Spotsylvania|11.1   |4.9209481351555455|
|Hanover     |1.5    |5.3908355795148255|
|Henrico     |2.8    |7.605108499095841 |
|Loudoun     |0.8    |3.2100979202925304|
+------------+-------+------------------+



In [33]:
from pyspark.sql.functions import when

joined_with_bucket = joined.withColumn(
    "poverty_bucket",
    when(joined.poverty_percent < 5, "0–5%")
    .when((joined.poverty_percent >= 5) & (joined.poverty_percent < 10), "5–10%")
    .when((joined.poverty_percent >= 10) & (joined.poverty_percent < 20), "10–20%")
    .otherwise("20%+")
)

bucket_summary = joined_with_bucket.groupBy("poverty_bucket") \
    .agg({"od_rate": "avg"}) \
    .withColumnRenamed("avg(od_rate)", "avg_od_rate")

bucket_summary.show(truncate=False)

+--------------+-----------+
|poverty_bucket|avg_od_rate|
+--------------+-----------+
|5–10%         |19.1625    |
|0–5%          |11.7875    |
+--------------+-----------+



In [36]:
# Example: replace these with the real BA+ column codes you identify
ba_cols = [
    "B17018_00AE",  # placeholder
    "B17018_00BE"   # placeholder
    # add all BA+ columns you decide to include
]

from pyspark.sql.functions import sum as _sum

# Cast and sum BA+ columns
acs_ba = acs_va_clean

for c in ba_cols:
    acs_ba = acs_ba.withColumn(c, acs_ba[c].cast("double"))

acs_ba = acs_ba.withColumn(
    "ba_plus_families",
    sum(acs_ba[c] for c in ba_cols)  # or build with reduce/_sum if needed
)

acs_ba = acs_ba.withColumn(
    "ba_plus_percent",
    when(acs_ba.total_families > 0, acs_ba.ba_plus_families / acs_ba.total_families * 100).otherwise(None)
)

acs_join_ed = acs_ba.select("county_clean", "poverty_percent", "ba_plus_percent")

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `B17018_00AE` cannot be resolved. Did you mean one of the following? [`GEO_ID`, `NAME`, `total_families`, `poverty_families`, `poverty_percent`, `county_clean`]. SQLSTATE: 42703

In [39]:
from pyspark.sql.functions import when

joined_with_pov_bucket = joined.withColumn(
    "poverty_bucket",
    when(joined.poverty_percent < 5, "0–5%")
    .when((joined.poverty_percent >= 5) & (joined.poverty_percent < 10), "5–10%")
    .when((joined.poverty_percent >= 10) & (joined.poverty_percent < 20), "10–20%")
    .otherwise("20%+")
)

poverty_summary = joined_with_pov_bucket.groupBy("poverty_bucket") \
    .agg({"od_rate": "avg"}) \
    .withColumnRenamed("avg(od_rate)", "avg_od_rate")

poverty_summary.show(truncate=False)

print("Corr(poverty_percent, od_rate):",
      joined.stat.corr("poverty_percent", "od_rate"))

+--------------+-----------+
|poverty_bucket|avg_od_rate|
+--------------+-----------+
|5–10%         |19.1625    |
|0–5%          |11.7875    |
+--------------+-----------+

Corr(poverty_percent, od_rate): 0.22225605266859233


In [40]:
corr_pov_od = joined.stat.corr("poverty_percent", "od_rate")
print("Corr(poverty_percent, od_rate):", corr_pov_od)

Corr(poverty_percent, od_rate): 0.22225605266859233


In [42]:
# Locality-level data
joined.select("Locality", "od_rate", "poverty_percent") \
    .coalesce(1) \
    .write.mode("overwrite").option("header", "true") \
    .csv("locality_poverty_od")

# Bucket summary
poverty_summary.coalesce(1) \
    .write.mode("overwrite").option("header", "true") \
    .csv("poverty_bucket_od")